# Data cleaning & missing-value handling — corrected pipeline

**Goal of this notebook:** turn `train_V2.csv` and `score.csv` into two clean, model-ready
tables (`train_clean`, `score_clean`) with the *same* columns and transformations.

**Guiding principle:** the three outcome columns are pulled OFF the training set before any
transformation. That way nothing we do to the features can leak or accidentally rescale the
targets, and the final profit predictions stay in euros.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
pd.set_option('display.max_columns', None)

## 1. Load data and separate the outcomes

In [2]:
train = pd.read_csv('train_V2.csv')
score = pd.read_csv('score.csv')

OUTCOME_COLS = ['outcome_profit', 'outcome_damage_inc', 'outcome_damage_amount']

# Keep the targets aside, untouched, indexed so we can re-attach later.
y = train[OUTCOME_COLS].copy()

# Feature matrices only. Tag origin so we can split back after joint cleaning.
train_X = train.drop(columns=OUTCOME_COLS).copy()
train_X['is_train'] = 1
score_X = score.copy()
score_X['is_train'] = 0

datafull = pd.concat([train_X, score_X], axis=0, ignore_index=True)
print('train:', train_X.shape, '| score:', score_X.shape, '| combined:', datafull.shape)

train: (5000, 51) | score: (500, 51) | combined: (5500, 51)


## 2. Survey the missingness

We decide *which columns to drop* using the TRAINING rows only — we never let the applicants
influence a modelling choice.

In [3]:
train_part = datafull[datafull['is_train'] == 1]
miss = (train_part.isnull().mean() * 100).sort_values(ascending=False)
miss[miss > 0].round(1)

score2_pos          75.8
score4_pos          75.5
score1_pos          75.5
score5_pos          75.4
score3_pos          74.8
score2_neg          73.9
score1_neg          73.7
score4_neg          73.5
score3_neg          72.7
score5_neg          70.1
tenure_mts           7.8
tenure_yrs           7.8
neighbor_income      4.8
shop_use             1.8
dining_ic            1.8
cab_requests         1.8
presidential         1.8
profit_am            1.1
income_am            1.1
client_segment       1.1
nights_booked        1.1
claims_no            1.1
company_ic           1.1
bar_no               1.1
sport_ic             1.1
marketing_permit     1.1
age                  1.1
gluten_ic            1.1
credit_use_ic        1.1
lactose_ic           1.1
insurance_ic         1.1
crd_lim_rec          1.1
damage_inc           1.1
profit_last_am       1.1
sect_empl            1.1
gold_status          1.1
urban_ic             1.1
prev_stay            1.1
retired              1.1
fam_adult_size       1.1


### 2a. The staff-score columns are *block*-missing, not randomly missing

All ten `scoreX_pos` / `scoreX_neg` columns are ~70–76% missing. Crucially, the missingness is
structured: a guest tends to be missing *all* their staff scores at once. That almost certainly
means "this guest was never rated by partner hotels" — a meaningful fact, not noise.

So before we drop these columns, we capture the signal as a single feature: how many staff
scores the guest actually has.

In [4]:
score_cols = [c for c in datafull.columns if c.startswith('score')]

# How concentrated is the missingness? (computed on train for reporting)
per_row = train_part[score_cols].isnull().sum(axis=1)
print('Guests missing ALL 10 staff scores:', (per_row == 10).sum())
print('Guests missing NONE:', (per_row == 0).sum())

# Engineered feature: number of staff scores present (0..10). Keeps the information
# that the raw columns are about to lose.
datafull['n_staff_scores'] = datafull[score_cols].notnull().sum(axis=1)

# Now drop the raw score columns (too sparse to impute responsibly).
datafull = datafull.drop(columns=score_cols)
print('Dropped', len(score_cols), 'sparse score columns; kept n_staff_scores instead.')

Guests missing ALL 10 staff scores: 1166
Guests missing NONE: 10
Dropped 10 sparse score columns; kept n_staff_scores instead.


## 3. Remove a redundant feature

`tenure_mts` and `tenure_yrs` are the same thing in different units (correlation ≈ 0.9998,
months ≈ 12 × years). Keeping both adds nothing. We keep months (finer granularity) and drop years.

In [5]:
datafull = datafull.drop(columns=['tenure_yrs'])

## 4. Encode categorical variables

- `gender` (M / V) and `married_cd` (True/False) are binary → 0/1.
- `client_segment` and `sect_empl` are nominal with several levels → one-hot.
  We KEEP all dummy levels (`drop_first=False`) because the models are tree-based
  gradient boosting, which handles redundant dummies fine and reads them more naturally.
- `dummy_na=True` turns "missing segment" into its own informative category.

In [6]:
# Binary
datafull['married_cd'] = datafull['married_cd'].astype(int)
datafull['gender'] = datafull['gender'].map({'M': 0, 'V': 1})  # NaN stays NaN -> imputed later

# Nominal -> one-hot (missing kept as its own column)
for col in ['client_segment', 'sect_empl']:
    datafull[col] = datafull[col].astype('category')
datafull = pd.get_dummies(datafull, columns=['client_segment', 'sect_empl'],
                          drop_first=False, dummy_na=True)

# any bool dummies -> int
bool_cols = datafull.select_dtypes('bool').columns
datafull[bool_cols] = datafull[bool_cols].astype(int)

## 5. Feature engineering

A simple, defensible example: profit per night (normalises a big spender who simply stayed
longer vs. one who is genuinely high-value per night). Log transforms tame the extreme
right-skew of the money variables (skew was ~20–36) so models aren't dominated by a few
outliers. `log1p` is safe at zero.

In [7]:
# Profit per night (guard against divide-by-zero)
datafull['profit_per_night'] = datafull['profit_am'] / datafull['nights_booked'].replace(0, np.nan)

# Log-transform heavily skewed positive money features
skewed_money = ['income_am', 'profit_am', 'profit_last_am', 'damage_am',
                'claims_am', 'shop_am', 'neighbor_income', 'crd_lim_rec']
for col in skewed_money:
    if col in datafull.columns:
        datafull[col + '_log'] = np.log1p(datafull[col].clip(lower=0))

## 6. Outlier handling 

The exam asks us to *identify* outliers and make a deliberate choice. We don't delete rows —
the applicants need a score, and dropping training rows throws away signal. Instead we
**winsorize** (cap) the most extreme numeric values at the 1st/99th percentile, computed on
the training rows only. This makes the analysis robust without losing any guest.

In [8]:
# Numeric continuous columns to cap (exclude binaries/dummies/flags/engineered logs are fine to cap too)
cap_cols = ['income_am', 'profit_am', 'profit_last_am', 'damage_am', 'claims_am',
            'shop_am', 'neighbor_income', 'crd_lim_rec', 'age', 'nights_booked',
            'cab_requests', 'bar_no', 'profit_per_night']
cap_cols = [c for c in cap_cols if c in datafull.columns]

train_mask = datafull['is_train'] == 1
for col in cap_cols:
    lo = datafull.loc[train_mask, col].quantile(0.01)
    hi = datafull.loc[train_mask, col].quantile(0.99)
    datafull[col] = datafull[col].clip(lower=lo, upper=hi)

## 7. Impute remaining missing values

Type-aware: binary columns get the mode, all other numerics get the **median** (robust to the
remaining skew). Stats are taken from the TRAINING rows so the applicants never influence them.

In [9]:
# After encoding, the genuinely binary columns:
binary_vars = ['gender', 'married_cd', 'gold_status', 'marketing_permit', 'urban_ic',
               'prev_stay', 'prev_all_in_stay', 'divorce', 'retired', 'company_ic']
binary_vars = [c for c in binary_vars if c in datafull.columns]

feature_cols = [c for c in datafull.columns if c != 'is_train']

for col in feature_cols:
    if datafull[col].isnull().any():
        if col in binary_vars:
            fill = datafull.loc[train_mask, col].mode()[0]
        else:
            fill = datafull.loc[train_mask, col].median()
        datafull[col] = datafull[col].fillna(fill)

print('Remaining missing values:', int(datafull[feature_cols].isnull().sum().sum()))

Remaining missing values: 0


## 8. Scale the features

We fit the scaler on the training rows and apply it to both, so the applicants are transformed
with exactly the train statistics (no leakage). Note: tree-based gradient boosting does not
*need* scaling, so this step is optional for that model family — but it is harmless and keeps
the pipeline ready for any linear/regularised model you might compare against.

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(datafull.loc[train_mask, feature_cols])
datafull[feature_cols] = scaler.transform(datafull[feature_cols])

## 9. Split back and re-attach the untouched outcomes

`is_train` is dropped here so it can never leak into a model.

In [11]:
train_clean = datafull[datafull['is_train'] == 1].drop(columns='is_train').reset_index(drop=True)
score_clean = datafull[datafull['is_train'] == 0].drop(columns='is_train').reset_index(drop=True)

# Re-attach the raw (un-scaled) outcomes to the training set only
train_clean = pd.concat([y.reset_index(drop=True), train_clean], axis=1)

print('train_clean:', train_clean.shape, '| score_clean:', score_clean.shape)
assert list(train_clean.drop(columns=OUTCOME_COLS).columns) == list(score_clean.columns), \
    'Feature columns must match between train and score!'
print('Feature columns match. Outcomes are in euros, untouched.')

train_clean.to_csv('train_clean.csv', index=False)
score_clean.to_csv('score_clean.csv', index=False)

train_clean: (5000, 64) | score_clean: (500, 61)
Feature columns match. Outcomes are in euros, untouched.


# Machine Learning Exam – Hotel Guest Selection

This notebook trains models to rank 500 hotel applicants by expected net value.

The final goal is to select 200 clients with an attractive balance between predicted revenue and expected damage costs.

In [12]:
## Imports and configuration

In [13]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    roc_auc_score, precision_score, recall_score, f1_score, brier_score_loss,
)

try:
    from xgboost import XGBRegressor, XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed - falling back to RandomForest only.")

RANDOM_SEED = 42
EXPM1_CAP = 11.5
TUNE_N_ITER = 20
TUNE_N_ITER_SMALL = 15
TUNE_CV = 5

xgboost not installed - falling back to RandomForest only.


## Helper functions

The following functions are used throughout the notebook for:

- evaluation metrics
- hyperparameter tuning
- probability calibration
- prediction post-processing

In [14]:
def safe_expm1(log_values):
    return np.expm1(np.clip(log_values, a_min=None, a_max=EXPM1_CAP))

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def trivial_brier(y_true):
    p = np.mean(y_true)
    return p * (1 - p)

def report_regression(name, y_true, y_pred):
    print(f"{name:26s} RMSE={rmse(y_true, y_pred):8.2f}  "
          f"MAE={mean_absolute_error(y_true, y_pred):8.2f}  "
          f"R2={r2_score(y_true, y_pred):.3f}")

def report_classification(name, y_true, proba, threshold=0.5):
    pred_label = (proba >= threshold).astype(int)
    print(f"{name:26s} AUC={roc_auc_score(y_true, proba):.3f}  "
          f"Brier={brier_score_loss(y_true, proba):.3f}  "
          f"Precision={precision_score(y_true, pred_label, zero_division=0):.3f}  "
          f"Recall={recall_score(y_true, pred_label, zero_division=0):.3f}  "
          f"MeanProba={np.mean(proba):.3f}")

Code: tuning functions

In [15]:
def make_regressor(n_estimators, max_depth, learning_rate=0.05):
    if HAS_XGB:
        return XGBRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_SEED,
            n_jobs=1
        )
    return RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth + 4,
        random_state=RANDOM_SEED,
        n_jobs=-1
    )


def tune_regressor(X, y, scoring="neg_root_mean_squared_error",
                   n_iter=TUNE_N_ITER, cv=TUNE_CV, label=""):
    if HAS_XGB:
        base = XGBRegressor(random_state=RANDOM_SEED, n_jobs=-1)
        param_dist = {
            "n_estimators": [100, 150, 200, 300, 400, 500],
            "max_depth": [2, 3, 4, 5, 6],
            "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08, 0.1],
            "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
            "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
        }
    else:
        base = RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=1)
        param_dist = {
            "n_estimators": [100, 200, 300, 400, 500],
            "max_depth": [4, 6, 8, 10, 12, None],
            "min_samples_leaf": [1, 2, 4, 8],
        }

    search = RandomizedSearchCV(
        base, param_dist, n_iter=n_iter, cv=cv, scoring=scoring,
        random_state=RANDOM_SEED, n_jobs=1
    )
    search.fit(X, y)

    print(f"[{label}] best params: {search.best_params_}")
    print(f"[{label}] best {cv}-fold CV score: {search.best_score_:.4f}")

    return search.best_estimator_


def tune_classifier(X, y, scale_pos_weight, scoring="roc_auc",
                    n_iter=TUNE_N_ITER, cv=TUNE_CV, label=""):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_SEED)

    if HAS_XGB:
        base = XGBClassifier(
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_SEED,
            n_jobs=-1
        )
        param_dist = {
            "n_estimators": [100, 150, 200, 300, 400, 500],
            "max_depth": [2, 3, 4, 5, 6],
            "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08, 0.1],
            "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
            "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
        }
    else:
        base = RandomForestClassifier(
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=1
        )
        param_dist = {
            "n_estimators": [100, 200, 300, 400, 500],
            "max_depth": [4, 6, 8, 10, 12, None],
            "min_samples_leaf": [1, 2, 4, 8],
        }

    search = RandomizedSearchCV(
        base, param_dist, n_iter=n_iter, cv=skf, scoring=scoring,
        random_state=RANDOM_SEED, n_jobs=1
    )
    search.fit(X, y)

    print(f"[{label}] best params: {search.best_params_}")
    print(f"[{label}] best {cv}-fold CV score: {search.best_score_:.4f}")

    return search.best_estimator_

## Load cleaned data

The preprocessing notebook created `train_clean.csv` and `score_clean.csv`.
This modelling notebook starts from those cleaned files.

In [16]:
train_clean = pd.read_csv("train_clean.csv")
score_clean = pd.read_csv("score_clean.csv")

OUTCOME_COLS = ["outcome_profit", "outcome_damage_inc", "outcome_damage_amount"]
feature_cols = [c for c in train_clean.columns if c not in OUTCOME_COLS]

X = train_clean[feature_cols]
y = train_clean[OUTCOME_COLS]

assert list(score_clean.columns) == feature_cols, "score_clean columns must match train features"

X_score = score_clean[feature_cols]

print(f"Train: {X.shape}, Score: {X_score.shape}")

Train: (5000, 61), Score: (500, 61)


Code: split

In [17]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y["outcome_damage_inc"]
)

print(f"Train split: {X_train.shape}, Val split: {X_val.shape}")

Train split: (4000, 61), Val split: (1000, 61)


## Model 1 – Expected Profit

The first model predicts expected hotel profit.  
A linear regression baseline is compared with a tuned tree-based model.

In [18]:
y_profit_train_log = np.log1p(y_train["outcome_profit"])
y_val_profit_actual = y_val["outcome_profit"].values

lr_profit = LinearRegression()
lr_profit.fit(X_train, y_profit_train_log)
pred_profit_lr = safe_expm1(lr_profit.predict(X_val))

print("Tuning profit model...")
profit_model = tune_regressor(
    X_train,
    y_profit_train_log,
    label="profit/train-split"
)

pred_profit_main = safe_expm1(profit_model.predict(X_val))

print("\n--- Profit model validation ---")
report_regression("Linear baseline", y_val_profit_actual, pred_profit_lr)
report_regression("Main model", y_val_profit_actual, pred_profit_main)

Tuning profit model...
[profit/train-split] best params: {'n_estimators': 500, 'min_samples_leaf': 1, 'max_depth': None}
[profit/train-split] best 5-fold CV score: -0.2724

--- Profit model validation ---
Linear baseline            RMSE=  704.41  MAE=  433.26  R2=0.762
Main model                 RMSE=  667.67  MAE=  409.28  R2=0.786


Code: leakage check

In [19]:
leak_cols = [c for c in ["profit_per_night", "profit_am", "profit_am_log"] if c in feature_cols]
reduced_cols = [c for c in feature_cols if c not in leak_cols]

print(f"Leakage check without: {leak_cols}")

profit_model_noleak = make_regressor(n_estimators=300, max_depth=4)
profit_model_noleak.fit(X_train[reduced_cols], y_profit_train_log)

pred_profit_noleak = safe_expm1(
    profit_model_noleak.predict(X_val[reduced_cols])
)

report_regression("Without profit history", y_val_profit_actual, pred_profit_noleak)

Leakage check without: ['profit_per_night', 'profit_am', 'profit_am_log']
Without profit history     RMSE= 1012.51  MAE=  484.36  R2=0.509


## Model 2 – Damage Probability

The second model predicts the probability that a guest causes damage.  
Because class weighting can distort probabilities, the model is calibrated using isotonic calibration.

In [20]:
y_dmg_train = y_train["outcome_damage_inc"]
y_dmg_val = y_val["outcome_damage_inc"]

base_rate_train = y_dmg_train.mean()

logit = LogisticRegression(max_iter=1000, class_weight="balanced")
logit.fit(X_train, y_dmg_train)

proba_dmg_lr = logit.predict_proba(X_val)[:, 1]

pos = y_dmg_train.sum()
neg = len(y_dmg_train) - pos

print("Tuning damage probability model...")

damage_model_raw = tune_classifier(
    X_train,
    y_dmg_train,
    scale_pos_weight=neg / pos,
    label="damage/train-split"
)

proba_dmg_raw = damage_model_raw.predict_proba(X_val)[:, 1]

print(f"\nTraining damage base rate: {base_rate_train:.3f}")
print(f"Trivial Brier score: {trivial_brier(y_dmg_val):.3f}")

report_classification("Logistic baseline", y_dmg_val, proba_dmg_lr)
report_classification("Main model raw", y_dmg_val, proba_dmg_raw)

Tuning damage probability model...
[damage/train-split] best params: {'n_estimators': 400, 'min_samples_leaf': 8, 'max_depth': 4}
[damage/train-split] best 5-fold CV score: 0.6738

Training damage base rate: 0.256
Trivial Brier score: 0.190
Logistic baseline          AUC=0.687  Brier=0.222  Precision=0.388  Recall=0.612  MeanProba=0.470
Main model raw             AUC=0.668  Brier=0.231  Precision=0.387  Recall=0.553  MeanProba=0.481


Code: calibration

In [21]:
damage_model_cal = CalibratedClassifierCV(
    clone(damage_model_raw),
    method="isotonic",
    cv=5
)

damage_model_cal.fit(X_train, y_dmg_train)

proba_dmg_main = damage_model_cal.predict_proba(X_val)[:, 1]

report_classification("Main calibrated", y_dmg_val, proba_dmg_main)

if brier_score_loss(y_dmg_val, proba_dmg_main) < trivial_brier(y_dmg_val):
    print("Calibrated Brier beats the trivial baseline.")
else:
    print("Warning: calibrated Brier does not beat the trivial baseline.")

Main calibrated            AUC=0.678  Brier=0.175  Precision=0.571  Recall=0.047  MeanProba=0.256
Calibrated Brier beats the trivial baseline.


Code: threshold sweep

In [22]:
print("--- Damage probability threshold sweep ---")

best_f1 = -1
best_thresh = 0.5

for t in np.arange(0.10, 0.60, 0.05):
    pred_label = (proba_dmg_main >= t).astype(int)

    p = precision_score(y_dmg_val, pred_label, zero_division=0)
    r = recall_score(y_dmg_val, pred_label, zero_division=0)
    f1 = f1_score(y_dmg_val, pred_label, zero_division=0)

    print(f"threshold={t:.2f} precision={p:.3f} recall={r:.3f} f1={f1:.3f}")

    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print(f"Best F1 threshold: {best_thresh:.2f}")

--- Damage probability threshold sweep ---
threshold=0.10 precision=0.260 recall=0.988 f1=0.412
threshold=0.15 precision=0.285 recall=0.886 f1=0.432
threshold=0.20 precision=0.324 recall=0.745 f1=0.452
threshold=0.25 precision=0.380 recall=0.561 f1=0.453
threshold=0.30 precision=0.423 recall=0.482 f1=0.451
threshold=0.35 precision=0.433 recall=0.392 f1=0.412
threshold=0.40 precision=0.503 recall=0.345 f1=0.409
threshold=0.45 precision=0.514 recall=0.216 f1=0.304
threshold=0.50 precision=0.571 recall=0.047 f1=0.087
threshold=0.55 precision=0.600 recall=0.035 f1=0.067
Best F1 threshold: 0.25


## Model 3 – Damage Amount

The third model predicts the amount of damage, conditional on damage occurring.  
Only observations with `outcome_damage_inc = 1` are used to train this model.

Code: cross-validation for damage amount

In [24]:
mask_train = y_train["outcome_damage_inc"] == 1
mask_val = y_val["outcome_damage_inc"] == 1

X_dmg_train = X_train[mask_train]
X_dmg_val = X_val[mask_val]

y_dmg_amt_train_log = np.log1p(
    y_train.loc[mask_train, "outcome_damage_amount"]
)

y_dmg_amt_val_actual = y_val.loc[
    mask_val,
    "outcome_damage_amount"
].values

print(f"Damage rows train: {X_dmg_train.shape[0]}")
print(f"Damage rows validation: {X_dmg_val.shape[0]}")

lr_amt = LinearRegression()
lr_amt.fit(X_dmg_train, y_dmg_amt_train_log)

pred_amt_lr = safe_expm1(lr_amt.predict(X_dmg_val))

print("Tuning damage amount model...")

amount_model = tune_regressor(
    X_dmg_train,
    y_dmg_amt_train_log,
    n_iter=TUNE_N_ITER_SMALL,
    label="damage-amount/train-split"
)

pred_amt_main = safe_expm1(amount_model.predict(X_dmg_val))

print("\n--- Damage amount validation ---")
report_regression("Linear baseline", y_dmg_amt_val_actual, pred_amt_lr)
report_regression("Main model", y_dmg_amt_val_actual, pred_amt_main)

Damage rows train: 1022
Damage rows validation: 255
Tuning damage amount model...
[damage-amount/train-split] best params: {'n_estimators': 200, 'min_samples_leaf': 8, 'max_depth': 6}
[damage-amount/train-split] best 5-fold CV score: -0.4813

--- Damage amount validation ---
Linear baseline            RMSE= 3910.42  MAE=  463.38  R2=-123.963
Main model                 RMSE=  319.65  MAE=  227.80  R2=0.165


In [25]:
dmg_mask_all = y["outcome_damage_inc"] == 1

X_dmg_all = X[dmg_mask_all]
y_dmg_amt_all_log = np.log1p(
    y.loc[dmg_mask_all, "outcome_damage_amount"]
)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

cv_scores = []

for fold, (tr_idx, te_idx) in enumerate(kf.split(X_dmg_all), 1):
    m = clone(amount_model)

    m.fit(
        X_dmg_all.iloc[tr_idx],
        y_dmg_amt_all_log.iloc[tr_idx]
    )

    pred = safe_expm1(m.predict(X_dmg_all.iloc[te_idx]))
    actual = np.expm1(y_dmg_amt_all_log.iloc[te_idx])

    fold_rmse = rmse(actual, pred)
    cv_scores.append(fold_rmse)

    print(f"Fold {fold}: RMSE = {fold_rmse:.2f}")

print(f"5-fold CV RMSE: {np.mean(cv_scores):.2f} +/- {np.std(cv_scores):.2f}")

Fold 1: RMSE = 330.77
Fold 2: RMSE = 385.13
Fold 3: RMSE = 319.09
Fold 4: RMSE = 382.69
Fold 5: RMSE = 408.00
5-fold CV RMSE: 365.14 +/- 34.19


## Final model training

After validation, the final models are trained on all available training data.
These final models are used to score the 500 applicants.

In [26]:
print("Tuning final profit model...")

final_profit_model = tune_regressor(
    X,
    np.log1p(y["outcome_profit"]),
    label="profit/final-full-data"
)

pos_all = y["outcome_damage_inc"].sum()
neg_all = len(y) - pos_all

print("\nTuning final damage probability model...")

damage_model_final_raw = tune_classifier(
    X,
    y["outcome_damage_inc"],
    scale_pos_weight=neg_all / pos_all,
    label="damage/final-full-data"
)

final_damage_model = CalibratedClassifierCV(
    clone(damage_model_final_raw),
    method="isotonic",
    cv=5
)

final_damage_model.fit(X, y["outcome_damage_inc"])

print("\nTuning final damage amount model...")

final_amount_model = tune_regressor(
    X_dmg_all,
    y_dmg_amt_all_log,
    n_iter=TUNE_N_ITER_SMALL,
    label="damage-amount/final-full-data"
)

print("Final models trained.")

Tuning final profit model...
[profit/final-full-data] best params: {'n_estimators': 500, 'min_samples_leaf': 1, 'max_depth': None}
[profit/final-full-data] best 5-fold CV score: -0.2771

Tuning final damage probability model...
[damage/final-full-data] best params: {'n_estimators': 200, 'min_samples_leaf': 8, 'max_depth': 6}
[damage/final-full-data] best 5-fold CV score: 0.6749

Tuning final damage amount model...
[damage-amount/final-full-data] best params: {'n_estimators': 100, 'min_samples_leaf': 8, 'max_depth': 8}
[damage-amount/final-full-data] best 5-fold CV score: -0.4955
Final models trained.


## Score applicants and select final 200

Expected net value is calculated as:

`expected_net_value = expected_profit - (p_damage × expected_damage_amount)`

The final selected clients are the 200 applicants with the highest expected net value.

In [27]:
expected_profit = safe_expm1(final_profit_model.predict(X_score))
p_damage = final_damage_model.predict_proba(X_score)[:, 1]
expected_damage_amount = safe_expm1(final_amount_model.predict(X_score))

expected_damage_cost = p_damage * expected_damage_amount
expected_net_value = expected_profit - expected_damage_cost

results = score_clean.copy()

results["predicted_revenue"] = expected_profit
results["p_damage"] = p_damage
results["expected_damage_amount"] = expected_damage_amount
results["predicted_damage_costs"] = expected_damage_cost
results["overall_predicted_revenue"] = expected_net_value

results = results.sort_values(
    "overall_predicted_revenue",
    ascending=False
).reset_index(drop=True)

selected_200 = results.head(200).copy()

selected_200["predicted_damage_status"] = np.where(
    selected_200["p_damage"] >= best_thresh,
    "yes",
    "no"
)

selected_200.to_csv("selected_200_clients.csv", index=False)

print(f"Selected {len(selected_200)} clients.")
selected_200.head()

Selected 200 clients.


,income_am,profit_last_am,profit_am,damage_am,damage_inc,crd_lim_rec,credit_use_ic,gluten_ic,lactose_ic,insurance_ic,spa_ic,empl_ic,cab_requests,married_cd,bar_no,sport_ic,neighbor_income,age,marketing_permit,urban_ic,dining_ic,presidential,prev_stay,prev_all_in_stay,divorce,fam_adult_size,children_no,tenure_mts,company_ic,claims_no,claims_am,nights_booked,gender,shop_am,shop_use,retired,gold_status,n_staff_scores,client_segment_0.0,client_segment_1.0,client_segment_2.0,client_segment_3.0,client_segment_4.0,client_segment_5.0,client_segment_nan,sect_empl_0.0,sect_empl_1.0,sect_empl_2.0,sect_empl_3.0,sect_empl_4.0,sect_empl_6.0,sect_empl_nan,profit_per_night,income_am_log,profit_am_log,profit_last_am_log,damage_am_log,claims_am_log,shop_am_log,neighbor_income_log,crd_lim_rec_log,predicted_revenue,p_damage,expected_damage_amount,predicted_damage_costs,overall_predicted_revenue,predicted_damage_status
0,4.355186,-0.389747,6.142429,-0.30641,0.735739,-0.738364,-0.206241,-0.158146,-0.320592,-0.794222,-0.815476,-0.157481,-1.323665,0.482118,-0.365048,-0.629800,-0.929886,0.069861,1.019796,0.360122,-0.225525,-0.064944,0.349763,-0.577350,-0.33592,1.296317,-0.459390,-0.607622,-0.136912,6.748451,-0.163002,-0.686339,-0.982945,-0.319981,-0.418765,-0.468839,5.298091,0.671251,-0.261051,-1.450401,-0.448394,4.075001,-0.128323,-0.086343,-0.103506,-2.638519,3.247667,-0.092039,-0.024502,-0.072299,-0.120019,-0.103506,5.7515,2.634317,3.893952,0.395595,-0.404822,-0.191142,-0.405653,-0.719784,-1.096091,13004.558916,0.123903,786.028743,97.391412,12907.167504,no
1,3.809357,0.334851,6.142429,-0.30641,-0.393734,-0.738364,-0.206241,-0.158146,-0.320592,-0.794222,-0.815476,-0.157481,0.971121,-2.074181,-0.843800,-0.629800,1.363894,1.935808,1.019796,-2.776833,-0.225525,-0.064944,0.349763,-0.577350,-0.33592,-1.199961,-0.459390,-1.686978,-0.136912,-0.304696,-0.163002,-0.770529,1.017350,-0.319981,-0.418765,2.132930,-0.188747,0.671251,-0.261051,-1.450401,-0.448394,4.075001,-0.128323,-0.086343,-0.103506,0.379000,-0.307913,-0.092039,-0.024502,-0.072299,-0.120019,-0.103506,5.7515,2.539619,3.698018,1.051950,-0.404822,-0.191142,-0.405653,1.021935,-1.096091,9272.914198,0.133404,613.537520,81.848176,9191.066022,no
2,4.988629,-0.020447,4.098467,-0.30641,-0.393734,0.853992,-0.206241,-0.158146,-0.320592,-0.794222,-0.815476,-0.157481,-0.995839,0.482118,0.592455,1.587806,-0.927135,1.562619,1.019796,-2.776833,-0.225525,-0.064944,0.349763,-0.577350,-0.33592,-1.199961,-0.459390,0.758650,-0.136912,-0.304696,-0.163002,-0.686339,-0.982945,-0.319981,-0.418765,2.132930,5.298091,0.671251,-0.261051,-1.450401,-0.448394,4.075001,-0.128323,-0.086343,-0.103506,0.379000,-0.307913,-0.092039,-0.024502,-0.072299,-0.120019,-0.103506,5.7515,2.731601,3.154035,0.873662,-0.404822,-0.191142,-0.405653,-0.717185,1.006674,9181.684454,0.202673,717.618206,145.441861,9036.242593,no
3,5.799173,-0.328489,4.065183,-0.30641,-0.393734,-0.738364,-0.206241,-0.158146,3.119233,-0.794222,-0.815476,-0.157481,-0.340186,0.482118,0.592455,-0.629800,0.856466,0.256456,1.019796,0.360122,-0.225525,-0.064944,0.349763,1.732051,-0.33592,1.296317,3.157854,-0.061113,-0.136912,-0.304696,-0.163002,-0.686339,-0.982945,-0.319981,-0.418765,-0.468839,5.298091,0.671251,-0.261051,-1.450401,-0.448394,4.075001,-0.128323,-0.086343,-0.103506,-2.638519,3.247667,-0.092039,-0.024502,-0.072299,-0.120019,-0.103506,5.7515,2.840772,3.143833,0.540613,-0.404822,-0.191142,-0.405653,0.695125,-1.096091,9188.665179,0.438920,863.566728,379.037042,8809.628138,yes
4,0.387268,-0.247102,0.918358,-0.30641,1.865211,2.673828,-0.206241,6.323259,3.119233,1.259093,1.226277,-0.157481,0.643294,0.482118,4.183092,1.587806,1.373117,0.069861,-0.980588,0.360122,-0.225525,-0.064944,0.349763,1.732051,-0.33592,1.296317,1.952106,1.400798,-0.136912,-0.304696,-0.163002,-0.770529,-0.982945,2.566316,2.387973,-0.468839,5.298091,0.671251,-0.261051,-1.450401,2.230183,-0.245399,-0.128323,-0.086343,-0.103506,0.379000,-0.307913,-0.092039,-0.024502,-0.072299,-0.120019,-0.103506,5.7515,

Code: sanity checks

In [28]:
print("--- Sanity checks ---")

print("Any negative predicted revenue?", bool((expected_profit < 0).any()))

print(f"Training base rate for damage: {y['outcome_damage_inc'].mean():.3f}")
print(f"Scored-batch mean p_damage: {p_damage.mean():.3f}")

print("p_damage range:", round(p_damage.min(), 3), "-", round(p_damage.max(), 3))

print("\nSummary of selected 200 clients:")
print(
    selected_200[
        [
            "predicted_revenue",
            "p_damage",
            "expected_damage_amount",
            "predicted_damage_costs",
            "overall_predicted_revenue"
        ]
    ].describe()
)

--- Sanity checks ---
Any negative predicted revenue? False
Training base rate for damage: 0.255
Scored-batch mean p_damage: 0.258
p_damage range: 0.006 - 0.761

Summary of selected 200 clients:
       predicted_revenue    p_damage  expected_damage_amount  \
count         200.000000  200.000000              200.000000   
mean         2568.626750    0.229346              650.116867   
std          1440.945288    0.115168              154.871812   
min          1694.488889    0.005546              269.763997   
25%          1926.890556    0.155699              536.324003   
50%          2106.752625    0.212098              627.070331   
75%          2650.792736    0.279994              722.046398   
max         13004.558916    0.761389             1138.002360   

       predicted_damage_costs  overall_predicted_revenue  
count              200.000000                 200.000000  
mean               153.218551                2415.408200  
std                103.279951                1408.5

Code: top/bottom applicants

In [29]:
print("Top 5 selected applicants:")
display(
    selected_200[
        [
            "predicted_revenue",
            "p_damage",
            "predicted_damage_status",
            "expected_damage_amount",
            "predicted_damage_costs",
            "overall_predicted_revenue"
        ]
    ].head()
)

print("Lowest-ranked selected applicants:")
display(
    selected_200[
        [
            "predicted_revenue",
            "p_damage",
            "predicted_damage_status",
            "expected_damage_amount",
            "predicted_damage_costs",
            "overall_predicted_revenue"
        ]
    ].tail()
)

Top 5 selected applicants:


,predicted_revenue,p_damage,predicted_damage_status,expected_damage_amount,predicted_damage_costs,overall_predicted_revenue
0,13004.558916,0.123903,no,786.028743,97.391412,12907.167504
1,9272.914198,0.133404,no,613.537520,81.848176,9191.066022
2,9181.684454,0.202673,no,717.618206,145.441861,9036.242593
3,9188.665179,0.438920,yes,863.566728,379.037042,8809.628138
4,9300.315399,0.761389,yes,1097.623938,835.718671,8464.596728


Lowest-ranked selected applicants:


,predicted_revenue,p_damage,predicted_damage_status,expected_damage_amount,predicted_damage_costs,overall_predicted_revenue
195,1907.535888,0.338778,yes,670.916987,227.291854,1680.244033
196,1777.379726,0.172415,no,627.140382,108.128436,1669.251290
197,1757.742681,0.136219,no,687.292271,93.622524,1664.120157
198,1694.488889,0.064842,no,474.138824,30.743981,1663.744908
199,1923.345942,0.354275,yes,735.132614,260.439337,1662.906605


Code: feature importance

In [30]:
if hasattr(final_profit_model, "feature_importances_"):
    importances = pd.Series(
        final_profit_model.feature_importances_,
        index=feature_cols
    )

    print("Top 10 features driving expected profit:")
    display(importances.sort_values(ascending=False).head(10))
else:
    print("Final profit model does not provide feature importances.")

Top 10 features driving expected profit:


profit_per_night       0.516349
tenure_mts             0.040757
age                    0.035337
cab_requests           0.029303
neighbor_income_log    0.026548
neighbor_income        0.025622
presidential           0.021752
bar_no                 0.021218
gender                 0.018203
damage_am_log          0.017299
dtype: float64